In [1]:
%load_ext autoreload
%autoreload 2

# Edits to sequence_graph/*.py take effect on the next cell without restarting the kernel.

# sequence_graph — exploration

`sequence_graph` is the project's spine (`docs/SEQUENCE_MODEL.md`;
`plans/SEQUENCE_GRAPH_SPINE.md`). It models three sequences —
**physical** (scans), **HTR** (sessions → resolutions → paragraphs), **editorial**
(sitting days → enriched resolutions) — connected by four cross-sequence edges:

| edge | from → to | observed? |
|---|---|---|
| `OBSERVED_ON` | Session / HtrResolution → Scan | yes |
| `DATED_AS` | Scan → SittingDay | yes (channel: strict / bootstrap) |
| `PINNED_TO` | SittingDay → Session | **no — inferred** |
| `ALIGNED_TO` | EnrichedResolution → HtrResolution | derived (Stage 4, `align.py`) |

**Status (2026-10-09):** Stages 0–3b (nodes/build, correspondence/classify, consolidate/query,
render) are done and gate-verified. Stage 4 (`align.py`, the DP aligner) has a first pass: one
scored bracket, unfitted cost, not yet a gate. This notebook walks through what each stage
actually produces, as a working surface for the open design questions from today's session
(§6) rather than as a finished report.

## 1. Build the graph once

In [2]:
import sys
from pathlib import Path
import pandas as pd
from IPython.display import display, HTML

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print('repo root:', ROOT)

repo root: /Users/rikhoekstra/develop/republic_ner_matching


In [3]:
from sequence_graph.build import build, counts_by_kind
from sequence_graph.correspondence import add_observed_on, add_dated_as, add_pinned_and_labelled

graph = build()
add_observed_on(graph)
dated_n, dated_report = add_dated_as(graph)
pinned_n, labelled_n, pinned_report, labelled_report = add_pinned_and_labelled(graph)

print('nodes by kind:', counts_by_kind(graph))
print('DATED_AS edges:', dated_n, dated_report)
print('PINNED_TO edges:', pinned_n, pinned_report)
print('LABELLED_AS edges:', labelled_n, labelled_report)

corpus scope: baseline ends 1630-05-14; dropped 116 anchors dated after it (not a population, not a denominator)
sidelined: dropped 44 anchors from ['4562', '4861'] (the secret series does not enter the sequence)
nodes by kind: {'Scan': 3195, 'Session': 1034, 'HtrResolution': 11273, 'Paragraph': 18946, 'SittingDay': 1595, 'EnrichedResolution': 19120}
DATED_AS edges: 930 MonotonicityReport(DATED_AS: 39/930 = 0.0419)
PINNED_TO edges: 1089 MonotonicityReport(PINNED_TO: 25/1089 = 0.0230)
LABELLED_AS edges: 1034 MonotonicityReport(LABELLED_AS: 69/1034 = 0.0667)


## 2. The `PINNED_TO` rule in practice — a competing-hypothesis date

`1629-05-18` is the hand-verified case the spine exists for: `running_axis`'s single
denormalised `resolved_session_id` pin has only one slot per date, so it cannot represent two
sessions both claiming the same day. The graph represents both without contradiction.

In [4]:
from sequence_graph.query import query_date

query_date('1629-05-18')

corpus scope: baseline ends 1630-05-14; dropped 116 anchors dated after it (not a population, not a denominator)
sidelined: dropped 44 anchors from ['4562', '4861'] (the secret series does not enter the sequence)
corpus scope: baseline ends 1630-05-14; dropped 116 anchors dated after it (not a population, not a denominator)
sidelined: dropped 44 anchors from ['4562', '4861'] (the secret series does not enter the sequence)
DATE 1629-05-18
  INFERRED path (SittingDay -> Session), inferred=True:
    LABELLED_AS: session-3188-num-79
  verdict:              contradicted
  problem_type:         T3_offset_over_7_days
  correspondence_class: bounded_gap
  implied window:       1629-03-23 .. 1629-03-25 (2 days)
  offset:               +54 days
  paragraphs:           11

    PINNED_TO: session-3188-num-129
  verdict:              reconciled
  problem_type:         T0_anchor
  correspondence_class: hard_anchor
  implied window:       1629-05-17 .. 1629-05-18 (1 days)
  paragraphs:           19



## 3. The two taxonomies — `consolidate.py`

`CORRESPONDENCE_CLASSES` (hard_anchor / bounded_gap / wide_window / unconstrained_gap) and
`PROBLEM_TYPES` (`T0`–`T6`) are each registered as a `check_partition`-verified taxonomy — both
must sum exactly to the whole, or `.check()` raises.

In [5]:
from sequence_graph.consolidate import build_consolidation

consolidation = build_consolidation()
consolidation.check()  # raises if either partition doesn't sum to the whole

print('correspondence classes:')
for name, tally in consolidation.correspondence_tallies().items():
    print(f'  {name:<20} {tally.value}')
print()
print('problem types:')
for name, tally in consolidation.problem_type_tallies().items():
    print(f'  {name:<20} {tally.value}')

corpus scope: baseline ends 1630-05-14; dropped 116 anchors dated after it (not a population, not a denominator)
sidelined: dropped 44 anchors from ['4562', '4861'] (the secret series does not enter the sequence)
corpus scope: baseline ends 1630-05-14; dropped 116 anchors dated after it (not a population, not a denominator)
sidelined: dropped 44 anchors from ['4562', '4861'] (the secret series does not enter the sequence)
correspondence classes:
  hard_anchor          323
  bounded_gap          406
  wide_window          224
  unconstrained_gap    81

problem types:
  T0_anchor            406
  T1_offset_1_2_days   127
  T2_offset_3_7_days   109
  T3_offset_over_7_days 170
  T4_anchor_bracket_suspect 26
  T5_window_too_wide_to_judge 115
  T6_no_scan_observed  81
  T7_nothing_inferred  0


## 4. Rendering — the three lanes

`Scan` / `SittingDay` / `Session`, correspondence as edges between them. A track segment with
no edge is a gap, not an omission. Pick an inventory below (3185–3189 are the main series).

In [6]:
from sequence_graph.consolidate import track_records
from sequence_graph.render import render_html

inventory = '3188'  # contains the 1629-05-18 fork from section 2
nodes, edges = track_records(inventory)
print(f'inventory {inventory}: {len(nodes)} nodes, {len(edges)} edges')
display(HTML(render_html(nodes, edges, title=f'Inventory {inventory}')))

corpus scope: baseline ends 1630-05-14; dropped 116 anchors dated after it (not a population, not a denominator)
sidelined: dropped 44 anchors from ['4562', '4861'] (the secret series does not enter the sequence)
inventory 3188: 1233 nodes, 1022 edges


## 5. Stage 4 first pass — the DAG aligner on one scoped bracket

`align.py` aligns one day's enriched resolutions (ref) against its HTR resolutions (read) via a
semi-global DP: `MATCH` / `DELETION_GAP` / `INSERTION_SPLIT` / `SOFT_CLIP`, free end gaps gated
by evidence (not position), abstention above a cost threshold. The cost model is **deliberately
unfitted** — see `CostModel`'s docstring. Scoped first run, per the user's decision: the single
cleanest gold day, `1629-03-02` (`k_e == k_f == 5`).

In [7]:
from sequence_graph.align import run_bracket

report = run_bracket('1629-03-02')
print(f"date: {report['date']}  k_e={report['k_e']}  k_f={report['k_f']}")
print(f"abstained: {report['abstained']}  total_cost: {report['total_cost']:.3f}")
for kind, ref_i, read_i in report['moves']:
    print(f'  {kind:<16} ref={ref_i}  read={read_i}')
print(f"separated_count: {report['separated_count']} / {report['k_e']}")
print(f"uniform-null start positions: {report['uniform_null_positions']}")
print(f"gold exact agreement: {report['exact_agreement']}")

date: 1629-03-02  k_e=5  k_f=5
abstained: False  total_cost: 5.489
  SOFT_CLIP        ref=None  read=0
  DELETION_GAP     ref=0  read=None
  MATCH            ref=1  read=1
  INSERTION_SPLIT  ref=None  read=2
  DELETION_GAP     ref=2  read=None
  MATCH            ref=3  read=3
  DELETION_GAP     ref=4  read=None
separated_count: 1 / 5
uniform-null start positions: [0, 1, 2, 3, 4]
gold exact agreement: 0.6


## 6. A spread of real days, not just the clean one

Per the user's explicit ask before this was recorded as a first pass (`docs/DECISIONS.md`
2026-10-09): run the same unfitted mechanism on days from clean through severe over/under-
segmentation and see whether cost and `separated_count` scale plausibly, rather than only
trusting the one hand-picked baseline. `boundary_gold_sample` days with zero HTR (`k_f == 0`,
structurally un-alignable) are shown separately, not fed through the DP.

In [8]:
from data_io import load

gold_days = load('boundary_gold_sample')['days']
candidates = sorted(
    (d['date'], len(d['enriched_ids']), len(d['flat_ids'])) for d in gold_days
)
alignable = [c for c in candidates if c[2] > 0]
alignable.sort(key=lambda c: abs(c[1] - c[2]))

# spread from clean (diff 0) to severe, a handful of representative days
sample_dates = [alignable[0][0], alignable[len(alignable) // 4][0],
                alignable[len(alignable) // 2][0], alignable[-1][0]]
print('no-HTR days (k_f == 0, not run):', [c[0] for c in candidates if c[2] == 0][:5], '...')
print('sampled for the DP:', sample_dates)

no-HTR days (k_f == 0, not run): ['1626-03-26', '1626-04-28', '1626-05-17', '1626-08-18', '1626-09-18'] ...
sampled for the DP: ['1628-01-01', '1627-12-07', '1626-01-08', '1626-11-16']


In [9]:
rows = []
for date in sample_dates:
    r = run_bracket(date)
    rows.append({
        'date': r['date'], 'k_e': r['k_e'], 'k_f': r['k_f'],
        'abstained': r['abstained'], 'total_cost': round(r['total_cost'], 3),
        'separated': r['separated_count'], 'gold_exact_agreement': r['exact_agreement'],
    })
pd.DataFrame(rows)

,date,k_e,k_f,abstained,total_cost,separated,gold_exact_agreement
0,1628-01-01,1,1,False,1.000,1,None
1,1627-12-07,7,5,False,4.150,0,None
2,1626-01-08,7,13,False,8.512,2,None
3,1626-11-16,14,47,False,28.207,4,None


## 7. Open design questions (2026-10-09) — working notes, not yet implemented

Raised after the Stage 4 first pass above, discussed live, recorded in `docs/DECISIONS.md`:

1. **Nihil-actum days are not a `SKIP` move.** User's correction: a nihil-actum day is a
   *milestone* — a session exists (HTR side), but it carries zero `EnrichedResolution`s —
   not a gap in the alignment that needs a special transition. It's an invariant to check and
   enforce (e.g. assert `n_ref == 0` whenever a day is nihil-actum), not a new DP move type.
   Quick sanity check below, read-only.
2. **`DATED_AS` channel (strict/bootstrap) stays upstream-only.** Decided: do not fold the
   scan→date parse-confidence into `align.py`'s per-bracket cost — it already gates which
   brackets get built before the solver runs, and folding it in a second time would double-
   count the same uncertainty.
3. **Track P (sub-paragraph division) should unify with `ALIGNED_TO` at paragraph granularity.**
   Decided: redesign the read sequence to be `Paragraph`/`CONTAINS`-child level, not
   `HtrResolution`-level, so a future sub-paragraph split is just a finer read item the same DP
   already handles — not a second alignment pass bolted on afterward. Not yet implemented;
   `align.py` still operates on `flat_id`s.
4. **"Forbid HTR insertion in the move set"** — intent not yet confirmed with the user; two
   readings were offered (restrict `SOFT_CLIP` to true margins only, vs. forbid orphan HTR
   content outright) and neither was it. Still open.

In [10]:
# Sanity check for point 1: does "nihil-actum" mean "session exists, zero enriched
# resolutions" in the data we have? This is read-only exploration for the design question
# above -- it does not change align.py. DateState.is_nihil checks status == NIHIL_STATUS
# directly; k_e is its own field, not derived here, so this is a real cross-check.
from collections import Counter
from running_axis.axis import RunningAxis

axis = RunningAxis.load()
nihil_dates = [d for d in axis.dates.values() if d.is_nihil]
print(f'{len(nihil_dates)} nihil-actum dates in running_axis')
print(f'  k_e distribution: {dict(Counter(d.k_e for d in nihil_dates))}')
has_session = sum(1 for d in nihil_dates if d.resolved_session_id)
print(f'  {has_session} have a resolved HTR session; {len(nihil_dates) - has_session} have none')

127 nihil-actum dates in running_axis
  k_e distribution: {1: 127}
  0 have a resolved HTR session; 127 have none


**Finding, not yet reconciled with the design conversation above:** every nihil-actum date
carries exactly `k_e == 1` — the formulaic "nihil actum" entry *is* the one enriched
resolution, not zero — and currently **none** of the 127 resolve to an HTR session in
`running_axis`. So the milestone shape in the live data is not yet "session exists, zero
resolutions"; it is "exactly one formulaic resolution, no pinned session at all" for every
case measured so far. This refines point 1 above rather than confirming it as stated — worth
raising back with the user before encoding either version as an invariant.